# 03 — Structured extraction and multi-task

**What you will learn**

- `POST /extract_structured`: declaring a record schema with the `::` field-spec
  syntax, and why the result is always a *list* of records.
- `POST /extract_multitask`: entities, a classification and a structured record
  from **one** forward pass.
- Why everything on the multi-task route nests under `schema_config`, and what
  the 400 looks like when you forget.
- `choices`: constraining a field to a closed vocabulary, and how it differs
  from a description.
- Two different schema syntaxes for what looks like the same thing, and why
  they are not interchangeable.

**What it assumes you already did**

[01 — Getting started](01-getting-started.ipynb) and
[02 — Extraction and classification](02-extraction-and-classification.ipynb).
You should be comfortable with entity extraction and with the idea that one
request can carry several tasks.

**Roughly how long**

About 25 minutes.

## Setup

In [1]:
import json
import os
import time

import requests

# Every notebook in this path reads the same environment variable, so you can
# point the whole series at a different deployment with one export:
#     export GLINER_BASE_URL=http://localhost:8013
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# The server bounds its own inference at REQUEST_TIMEOUT_SECONDS (default 120)
# and returns 504 when it blows through that. A client timeout slightly above
# the server's means the server always gets to explain itself with a status
# code instead of the client giving up first and leaving you guessing.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON. Raises on non-2xx."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises.

    Used whenever the interesting part of the answer IS the status code.
    """
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    """Pretty-print a JSON-serializable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

BASE_URL = http://192.168.1.177:8013


## Structured extraction — `POST /extract_structured`

Entity extraction gives you a bag of typed spans. Structured extraction gives
you **records**: named field sets filled from free text, which is what you
actually want when the downstream consumer is a table or a typed object rather
than a search index.

The schema is a JSON object. Each top-level key names a record type; its value
is a list of `::`-delimited field specifications:

```
"<field_name>::<dtype>"
"<field_name>::<dtype>::<description>"
```

`dtype` is `str` or `list`. The optional third segment is a description, and it
steers extraction the same way label descriptions did in notebook 02 — same
mechanism, same caveat that it is a hint rather than a rule.

The `::` syntax looks arbitrary next to the JSON around it, and it is: it is a
compact spelling that keeps a schema readable on one line. Notebook 03's
multi-task route uses a fully expanded JSON form for the same information, which
is the tradeoff — terse here, explicit there.

In [2]:
product = post("/extract_structured", {
    "text": "The Sony WH-1000XM5 headphones cost $399 and ship in 3 days.",
    "schema": {"product": ["name::str", "price::str", "shipping::str"]},
})
show(product)

{
  "product": [
    {
      "name": "Sony WH-1000XM5",
      "price": "$399",
      "shipping": "3 days"
    }
  ]
}


### The value is a list, and that is not an accident

Look closely: `product` maps to a *list* containing one record, not to the
record itself.

This catches people, because the obvious first test case — one product in one
sentence — makes the wrapping list look like pointless ceremony. It is not. A
schema describes a *shape that may occur*, and a document can contain that shape
zero, one, or many times. A press release might mention three products; a
transcript might contain four transactions. If the API returned a bare object
for the single case and a list otherwise, every client would need a type check
on every response.

So the rule is: **only index `[0]` when you already know the text holds exactly
one record**, and be honest with yourself about whether you know that. If the
text is user-supplied, you do not.

In [3]:
records = product["product"]
print(f"{len(records)} record(s)")
for i, rec in enumerate(records):
    print(f"  [{i}] {rec}")

1 record(s)
  [0] {'name': 'Sony WH-1000XM5', 'price': '$399', 'shipping': '3 days'}


Here is a text that genuinely holds more than one record, to make the point
concrete rather than theoretical:

In [4]:
multi_record = post("/extract_structured", {
    "text": ("The Sony WH-1000XM5 headphones cost $399 and ship in 3 days. "
             "The Bose QuietComfort Ultra costs $429 and ships in 5 days."),
    "schema": {"product": ["name::str", "price::str", "shipping::str"]},
})
show(multi_record)
print(f"\n{len(multi_record['product'])} record(s) extracted")

{
  "product": [
    {
      "name": "Sony WH-1000XM5",
      "price": "$399",
      "shipping": "3 days"
    },
    {
      "name": "Bose QuietComfort Ultra",
      "price": "$429",
      "shipping": "5 days"
    }
  ]
}

2 record(s) extracted


### Field descriptions

Same tool as label descriptions in notebook 02, applied to fields. Use it where
a bare field name is ambiguous *in context*: `"amount"` in a financial document
could be the trade value, a fee, or a share count, and the description settles
which one you meant.

In [5]:
show(post("/extract_structured", {
    "text": "Goldman Sachs processed a $2.5M equity trade for Tesla Inc.",
    "schema": {
        "transaction": [
            "broker::str::Financial institution",
            "amount::str::Transaction amount",
            "security::str::Stock name",
        ]
    },
}))

{
  "transaction": [
    {
      "broker": "Goldman Sachs",
      "amount": "$2.5M",
      "security": "Tesla Inc."
    }
  ]
}


## Multi-task — `POST /extract_multitask`

Everything so far has been one task per request. `/extract_multitask` runs
entities, a classification, a structured record — and, on a boundary checkpoint,
relations — over one text in **one forward pass**.

The saving is real and compounds. Three separate calls means encoding the same
text three times and passing through the inference semaphore three times. On a
box with one inference slot, that third fact dominates: you are not just doing
three times the model work, you are queueing three times. Notebook 06 measures
exactly this — 274.9 ms per document for three sequential single calls against
25.8 ms for one batched multi-task call over 32 documents.

### Everything nests under `schema_config`

There are no top-level `entities` / `classification` / `structure` keys on this
route. All of them go inside a `schema_config` object.

| Field | Shape |
|---|---|
| `schema_config.entities` | list of entity labels |
| `schema_config.classification` | `{"name": str, "labels": [str, ...]}` |
| `schema_config.structure` | `{"name": str, "fields": [{"name", "dtype", "description", "choices"}, ...]}` |
| `schema_config.relations` | list of relation names (boundary checkpoint only — notebook 05) |

Those four keys are the complete set. Anything else inside `schema_config` is a
**400** that names the allowed keys. All four are optional; supply at least one.

**Why the nesting?** Because `schema_config` is the *task specification*, and it
sits alongside a second, unrelated group of top-level keys: `text` (the input)
and the inference options from notebook 04 — `threshold`, `include_spans`,
`include_confidence`, `max_len`. Flattening the task keys into the top level
would mix "what to extract" with "how to run the model", and then the service
could no longer tell a mistyped task name from a mistyped option. The nesting is
what makes strict key validation possible in both groups independently.

In [6]:
multitask = post("/extract_multitask", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "schema_config": {
        "entities": ["company", "person", "location"],
        "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
        "structure": {
            "name": "announcement",
            "fields": [
                {"name": "who", "dtype": "str"},
                {"name": "what", "dtype": "str"},
            ],
        },
    },
})
show(multitask)

{
  "announcement": [
    {
      "who": "Tim Cook",
      "what": "record revenue"
    }
  ],
  "entities": {
    "company": [
      "Apple"
    ],
    "person": [
      "Tim Cook"
    ],
    "location": [
      "Cupertino"
    ]
  },
  "sentiment": "positive"
}


### Reading the response

The response is **flat**, and each task lands under a different key by a
different rule:

- entities go under the literal key `entities`;
- the classification goes under **the `name` you gave it** — `sentiment` here,
  because that is what you called the task;
- the structure goes under **its `name`**, as a *list* of records, following the
  same rule as `/extract_structured`.

So two of the three keys are chosen by you. That is a feature: you control your
own response schema, and it means a multi-task response merges cleanly into a
row you are building. It also means key ordering is not guaranteed — always
address by key, never by position.

In [7]:
print("entities      :", multitask["entities"])
print("sentiment     :", multitask["sentiment"])
print("announcement  :", multitask["announcement"][0])

entities      : {'company': ['Apple'], 'person': ['Tim Cook'], 'location': ['Cupertino']}
sentiment     : positive
announcement  : {'who': 'Tim Cook', 'what': 'record revenue'}


### The wrong shape, deliberately

This is the most common mistake with this endpoint: putting a task key at the
top level, where it looks like it belongs. Run it once so the failure is
recognizable when it turns up in a client log.

In [8]:
status, body = post_raw("/extract_multitask", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "entities": ["company", "person", "location"],   # WRONG: must nest under schema_config
})
print("HTTP", status)
show(body)

HTTP 400
{
  "detail": "Unknown payload key(s): ['entities']. Allowed: ['include_confidence', 'include_spans', 'max_len', 'overlap_policy', 'schema_config', 'text', 'threshold']."
}


In [9]:
# And a typo INSIDE schema_config - note the error lists the four allowed keys.
status, body = post_raw("/extract_multitask", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "schema_config": {"entitys": ["company", "person"]},   # WRONG: "entities"
})
print("HTTP", status)
show(body)

HTTP 400
{
  "detail": "Unknown schema_config key(s): ['entitys']. Allowed: ['classification', 'entities', 'relations', 'structure']."
}


Both of these used to be *silently ignored* in an earlier version of the
service. That is the change most likely to break an existing client, and it is
worth understanding why it was made rather than treating it as a regression.

Under the old behavior, a typo like `entitys` produced a perfectly plausible
`200` response that was quietly missing an entire task's worth of data. Nothing
in the response indicated a problem — the other tasks ran fine, the JSON parsed,
the pipeline kept going. You would only discover it downstream, weeks later,
when someone asked why the entity column was empty for a particular job.

A `400` fails loudly at the point of the mistake, in a message that names the
four legal keys. That is a much better trade than a plausible wrong answer.

## `choices`: constraining a field to a closed vocabulary

`structure.fields` entries accept a `choices` list, which restricts the field to
a fixed set instead of extracting freely from the text.

Compare it with `description`, since they occupy neighbouring slots and do
different jobs:

| | `description` | `choices` |
|---|---|---|
| What it does | Steers *what to look for* | Restricts *what may be returned* |
| Output | Free text pulled from the document | One of the values you listed |
| Strength | A hint | A closed answer space |
| Use when | The field name is ambiguous | The consumer expects an enum |

The important practical difference: `choices` values do **not** have to appear
in the text. The example below asks for `area` from `["frontend", "backend",
"infrastructure"]` against a ticket that says "checkout page returns a 500
error" — none of those three words appear anywhere in it. The model is
classifying into your vocabulary, not finding a substring.

That is exactly what you want when the field feeds a database column with a
`CHECK` constraint or a downstream enum, because free extraction would give you
whatever phrasing the document happened to use and you would spend the rest of
your life normalizing it.

In [10]:
show(post("/extract_multitask", {
    "text": "Support ticket: the checkout page returns a 500 error for all EU customers. "
            "This is blocking revenue and needs attention today.",
    "schema_config": {
        "entities": ["component", "error_code", "region"],
        "classification": {"name": "urgency", "labels": ["low", "medium", "high"]},
        "structure": {
            "name": "ticket",
            "fields": [
                {"name": "summary", "dtype": "str",
                 "description": "one-line problem statement"},
                {"name": "area", "dtype": "str",
                 "choices": ["frontend", "backend", "infrastructure"]},
            ],
        },
    },
}))

{
  "ticket": [
    {
      "summary": "the checkout page returns a 500 error",
      "area": "backend"
    }
  ],
  "entities": {
    "component": [
      "checkout page"
    ],
    "error_code": [
      "500"
    ],
    "region": [
      "EU"
    ]
  },
  "urgency": "high"
}


### Two schema syntaxes for the same idea

You have now seen the same structured schema written two ways, and it is worth
naming the difference explicitly because mixing them up is a reliable source of
400s.

**`/extract_structured` — compact `::` strings:**

```json
{"schema": {"announcement": ["who::str", "what::str::the thing announced"]}}
```

**`/extract_multitask` — expanded JSON objects, under `schema_config.structure`:**

```json
{"schema_config": {"structure": {
    "name": "announcement",
    "fields": [
        {"name": "who",  "dtype": "str"},
        {"name": "what", "dtype": "str", "description": "the thing announced"}
    ]}}}
```

They are not interchangeable — each route accepts only its own form. Note also
that the multi-task form carries the record name in an explicit `name` field
(because `structure` is a single object), where `/extract_structured` uses the
schema's own dictionary key (because it can carry several record types at once).

And there is one capability the compact form has no room for: **`choices` is
only expressible in the expanded form.** If you need a closed vocabulary, that
by itself is a reason to use `/extract_multitask` even when structure is the
only task you want.

## Try this yourself

Take a short job posting and pull a record out of it with a mix of free fields
and one constrained field. Predict, before running: for `seniority` with
`choices` of `["junior", "mid", "senior"]`, what happens when the posting says
"5+ years experience" and never uses any of those three words?

In [11]:
JOB = ("Hiring a Backend Engineer at Cloudflare in Austin. "
       "5+ years experience required. Salary 160k-190k, remote-friendly.")

show(post("/extract_multitask", {
    "text": JOB,
    "schema_config": {
        "entities": ["company", "location", "job_title"],
        "structure": {
            "name": "posting",
            "fields": [
                {"name": "title", "dtype": "str"},
                {"name": "salary_range", "dtype": "str"},
                {"name": "seniority", "dtype": "str",
                 "choices": ["junior", "mid", "senior"]},
                {"name": "work_mode", "dtype": "str",
                 "choices": ["onsite", "hybrid", "remote"]},
            ],
        },
    },
}))

{
  "posting": [
    {
      "title": "Backend Engineer",
      "salary_range": "160k-190k",
      "seniority": "junior",
      "work_mode": "remote"
    }
  ],
  "entities": {
    "company": [
      "Cloudflare"
    ],
    "location": [
      "Austin"
    ],
    "job_title": [
      "Backend Engineer"
    ]
  }
}


**Discussion.** The `choices` fields answer with one of your values even though
none of those strings appear in the posting — the model is mapping the document
onto your vocabulary rather than searching for a match. That is the whole point
of the mechanism, and it is what makes it safe to feed a typed column.
`work_mode` comes back `remote`, correctly picked up from "remote-friendly".

Now look at `seniority`. On this checkpoint it returns **`junior`** for a
posting that says "5+ years experience required" — which is wrong, and wrong
with no signal whatsoever that it is wrong. The response is shaped exactly like
the correct `work_mode` answer next to it.

That is the thing to internalize about `choices`: **a constrained field always
answers, and a guess is indistinguishable from a grounded answer.** There is no
"none of these" escape hatch unless you build one. Three defenses, in order of
how much they buy you:

1. **Put an explicit escape value in `choices`** — `["junior", "mid", "senior",
   "unspecified"]`. If "unspecified" is a legitimate outcome in your data, this
   is the single most useful habit with this feature.
2. **Add a `description`** to the field, spelling out the mapping you mean:
   `"junior is 0-2 years, mid is 3-6, senior is 7+"`. Constrained fields accept
   descriptions too, and this is exactly the ambiguity a description is for.
3. **Turn on `include_confidence`** (notebook 04) and route low-scoring
   constrained fields to review.

The free-text fields have the opposite problem. `salary_range` comes back
however the posting wrote it — `"160k-190k"` here, but `"$160,000 - $190,000"`
in the next document. Free extraction gives you fidelity to the source and hands
you the normalization problem. `choices` gives you a clean value and hands you
the risk of a confident guess. Choose per field, knowing which problem you would
rather have.

## What you learned

- `/extract_structured` takes `::`-delimited field specs and always returns a
  **list** of records per schema key, because a document can contain a shape
  more than once. Index `[0]` only when you know it cannot.
- Field descriptions steer extraction the same way label descriptions do.
- `/extract_multitask` runs entities, classification and structure in one
  forward pass — one encode, one trip through the inference semaphore.
- Everything nests under `schema_config`, whose four legal keys are `entities`,
  `classification`, `relations`, `structure`. Anything else is a **400** listing
  the allowed keys. The nesting is what keeps task keys and inference options
  independently validatable.
- The response is flat, and two of its three keys are names *you* chose.
- `choices` restricts a field to a closed vocabulary and does not require the
  values to appear in the text — but it always answers, so add an explicit
  "unspecified" value when that is a real outcome. It is only expressible in the
  multi-task schema form.

## Next

**[04 — Tuning and response shapes](04-tuning-and-response-shapes.ipynb)** —
`threshold`, `include_confidence`, `include_spans`, and what changes about your
parsing code when you turn them on.